[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-08-token-classification-ner.ipynb#scrollTo=1a2b3c4d)

---
# Day 8 · Token Classification — NER with IOB Tagging and Sequence Labeling Heads
**certified-journeys / huggingface-nlp-certified** · Day 8 · Token Classification

> **Goal for today:** Fine-tune BERT for Named Entity Recognition on CoNLL-2003, correctly aligning subword token labels using IOB2 tagging and evaluating with entity-level seqeval metrics.


In [ ]:
%pip install -q transformers datasets evaluate seqeval accelerate


## Step 1 · Load CoNLL-2003 and inspect the NER tag schema

CoNLL-2003 is the canonical NER benchmark. It annotates newswire text with four entity types:

| Entity type | Meaning | Example |
|---|---|---|
| PER | Person name | _Barack Obama_ |
| ORG | Organisation | _Google_, _UN_ |
| LOC | Location | _Paris_, _Amazon River_ |
| MISC | Miscellaneous | _World Cup_, _English_ (language) |

Labels are stored as integers; the `ClassLabel` feature maps them to readable strings.

**Reference:** [https://huggingface.co/docs/transformers/tasks/token_classification](https://huggingface.co/docs/transformers/tasks/token_classification)


In [ ]:
from datasets import load_dataset

# CoNLL-2003 is available under the 'conll2003' identifier
raw_datasets = load_dataset("conll2003", trust_remote_code=True)
print(raw_datasets)

# Inspect the label mapping: integers → IOB2 tag strings
label_names = raw_datasets["train"].features["ner_tags"].feature.names
print("\nNER tag label names:")
for i, name in enumerate(label_names):
    print(f"  {i}: {name}")

# Print a sample sentence with its NER tags
sample = raw_datasets["train"][4]
print("\nSample tokens:", sample["tokens"])
print("NER tag ids:  ", sample["ner_tags"])
print("NER tag names:", [label_names[t] for t in sample["ner_tags"]])


### What just happened?
- `load_dataset("conll2003")` fetches ~14k training sentences pre-tokenized at the word level.
- `features["ner_tags"].feature.names` gives the ordered list `['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']`.
- **Every sentence is a list of words** with a parallel list of integer tag ids — this word-level alignment is what we'll have to reconcile with BERT's subword tokenization.
- `O` (Outside) is the most common tag by far — NER data is highly class-imbalanced.


## Step 2 · Understand IOB2 tagging

IOB2 (Inside-Outside-Beginning) is the standard span encoding for NER:

```
Tokens:  Barack  Obama  visited  New    York   City  yesterday
IOB2:    B-PER   I-PER  O        B-LOC  I-LOC  I-LOC O
```

Rules:
- **B-** (Beginning): first token of a new entity span
- **I-** (Inside): continuation token of the same span type
- **O** (Outside): not part of any entity

Two adjacent spans of the same type **must** use B- at the boundary; otherwise they would merge into one span. This is why B- exists at all — `I-PER I-PER` would be ambiguous between one two-token span and two one-token spans.


In [ ]:
# Demonstrate IOB2 span decoding manually
example_tags = ["B-PER", "I-PER", "O", "B-LOC", "I-LOC", "I-LOC", "O"]
example_tokens = ["Barack", "Obama", "visited", "New", "York", "City", "yesterday"]

def decode_iob2_spans(tokens, tags):
    """Return a list of (entity_text, entity_type) tuples from IOB2 tags."""
    spans = []
    current_tokens = []
    current_type   = None

    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current_tokens:  # flush previous span
                spans.append((" ".join(current_tokens), current_type))
            current_tokens = [token]
            current_type   = tag[2:]          # strip 'B-' prefix
        elif tag.startswith("I-") and current_type == tag[2:]:
            current_tokens.append(token)      # continue existing span
        else:
            if current_tokens:                # O or mismatched I- → flush
                spans.append((" ".join(current_tokens), current_type))
            current_tokens = []
            current_type   = None

    if current_tokens:                        # flush final span
        spans.append((" ".join(current_tokens), current_type))

    return spans

spans = decode_iob2_spans(example_tokens, example_tags)
print("Decoded spans:")
for text, etype in spans:
    print(f"  [{etype}] {text!r}")


### What just happened?
- The decoder correctly groups `Barack Obama` as one `PER` span and `New York City` as one `LOC` span.
- **The B-/I- distinction is critical for span recovery** — without it, adjacent same-type entities would be indistinguishable.
- In practice, the `seqeval` library does this decoding internally when computing entity-level F1.
- `seqeval` expects predictions in tag-string format (e.g., `"B-PER"`), not integer ids — we'll convert back in the `compute_metrics` function.


## Step 3 · Tokenize and align labels with subword tokens

This is the core challenge of NER with subword tokenizers. BERT splits words like
`"playing"` → `["playing"]` or `"Obama"` → `["Obama"]`, but also
`"Wolfsberg"` → `["Wolf", "##sberg"]`.

The word label belongs to the **first subword** of each word. Continuation subwords (starting with `##`) should get label `-100` so PyTorch's `CrossEntropyLoss` ignores them.

Strategy using `word_ids()`:
- Iterate over the output token positions.
- If `word_id` is `None` → special token (`[CLS]`, `[SEP]`, `[PAD]`) → label `-100`.
- If `word_id == previous_word_id` → continuation subword → label `-100`.
- Otherwise → first subword of a new word → use the word's original label.


In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "bert-base-cased"  # cased matters for NER — 'Obama' ≠ 'obama'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    """
    Tokenize a batch of word-level examples and align NER labels to subword tokens.
    Continuation subwords receive label -100 (ignored by CrossEntropyLoss).
    """
    # is_split_into_words=True tells the tokenizer the input is already word-tokenized
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,   # critical: input is list-of-words, not a string
    )

    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned  = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                # [CLS], [SEP], or [PAD] — no entity, ignore in loss
                aligned.append(-100)
            elif word_id != prev_word_id:
                # First subword of a word → assign the word's true label
                aligned.append(word_labels[word_id])
            else:
                # Continuation subword (##...) → ignore in loss
                aligned.append(-100)

            prev_word_id = word_id

        all_labels.append(aligned)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

# Apply to all splits
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

# Verify alignment on the first training example
sample_tok = tokenized_datasets["train"][4]
tokens_decoded = tokenizer.convert_ids_to_tokens(sample_tok["input_ids"])
print("Token | Label")
for tok, lab in zip(tokens_decoded, sample_tok["labels"]):
    tag = label_names[lab] if lab != -100 else "[IGNORE]"
    print(f"  {tok:<15} {tag}")


### What just happened?
- `word_ids(batch_index=i)` maps each output token back to its source word index — `None` for special tokens.
- **Continuation subwords get `-100`** — PyTorch's `CrossEntropyLoss` skips these positions by default, so they don't push the model to predict a label for `##berg` or `##ing`.
- Only the **first subword** carries the word's true label; this means the model predicts entity presence at word boundaries, not at character boundaries.
- `is_split_into_words=True` is essential — without it, the tokenizer would treat the list as a sentence boundary list rather than individual words.


## Step 4 · Define `compute_metrics` with seqeval

`seqeval` computes entity-level (span-level) precision, recall, and F1 — not token-level accuracy. A predicted span only counts as correct if **both the span boundaries and the entity type** match the gold annotation. This is a stricter metric than token accuracy and better reflects real-world usefulness.


In [ ]:
import numpy as np
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # argmax over the tag dimension to get predicted label ids
    predictions = np.argmax(logits, axis=-1)

    # seqeval expects lists of tag strings, not integers
    # We must strip -100 (ignored) positions and convert ids → tag names
    true_labels = [
        [label_names[lab] for lab in seq if lab != -100]
        for seq in labels
    ]
    pred_labels = [
        [label_names[pred] for pred, lab in zip(pred_seq, lab_seq) if lab != -100]
        for pred_seq, lab_seq in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(
        predictions=pred_labels, references=true_labels
    )
    # Flatten to a simple dict for Trainer logging
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

print("compute_metrics function ready.")
print(f"Num NER labels: {len(label_names)}")


### What just happened?
- We filter out `-100` positions **in both** `true_labels` and `pred_labels` using the gold labels as the mask — predictions at `-100` positions are meaningless and should not be evaluated.
- **seqeval's entity-level F1** is the standard NER evaluation metric. Token accuracy can be misleadingly high because `O` tokens dominate; F1 on entity spans is the real signal.
- `results["overall_f1"]` aggregates across PER, ORG, LOC, MISC by token count (micro-averaged).
- Per-entity-type breakdown is available in `results["PER"]`, `results["ORG"]`, etc.


## Step 5 · Initialize the model and fine-tune with Trainer

`AutoModelForTokenClassification` adds a token-level linear head — one logit per tag per token — on top of BERT's encoder. `num_labels` must equal the number of IOB2 tags (9 for CoNLL-2003).


In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

num_labels = len(label_names)  # 9 for CoNLL-2003

# id2label / label2id make model.config aware of the tag meanings
# This is embedded in config.json so reloaded models know their own tags
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

print(f"Model: {MODEL_CHECKPOINT}")
print(f"Num labels: {model.config.num_labels}")
print(f"id2label: {model.config.id2label}")


In [ ]:
# DataCollatorForTokenClassification pads both input_ids AND labels
# It uses -100 as the pad label value, consistent with our alignment function
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./bert-ner-conll2003",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
)

# Use a small subset for demo speed — remove .select() for full training
small_train = tokenized_datasets["train"].select(range(1500))
small_eval  = tokenized_datasets["validation"].select(range(300))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(f"\nTraining done. Loss: {train_result.training_loss:.4f}")


### What just happened?
- `DataCollatorForTokenClassification` is the NER-specific collator — it pads the `labels` tensor alongside `input_ids` using `-100`, so variable-length sequences batch correctly.
- **Token classification loss** is per-token cross-entropy, averaged over non-`-100` positions. The model predicts one of 9 classes per token per sequence.
- On the full CoNLL-2003 train set, BERT-base-cased achieves ~91% entity F1 after 3 epochs — state of the art circa 2019.
- The seqeval F1 is computed at **entity span level** — partial boundary matches do not count.


In [ ]:
# Evaluate and run inference on a custom sentence
eval_results = trainer.evaluate()
print("Final eval results:")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Save the NER model
NER_SAVE_DIR = "./bert-ner-final"
trainer.save_model(NER_SAVE_DIR)
print(f"\nModel saved to {NER_SAVE_DIR}")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Reload and run inference on a custom sentence
ner_tokenizer = AutoTokenizer.from_pretrained(NER_SAVE_DIR)
ner_model     = AutoModelForTokenClassification.from_pretrained(NER_SAVE_DIR)
ner_model.eval()

test_text = "Elon Musk founded SpaceX in Hawthorne, California in 2002."

inputs  = ner_tokenizer(test_text, return_tensors="pt")
word_ids = inputs.word_ids()  # subword → word index mapping

with torch.no_grad():
    logits = ner_model(**inputs).logits  # shape: (1, seq_len, num_labels)

pred_ids = torch.argmax(logits, dim=-1)[0].tolist()  # (seq_len,)
tokens   = ner_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"{'Token':<20} {'Predicted Tag'}")
print("-" * 35)
for tok, pid, wid in zip(tokens, pred_ids, word_ids):
    if wid is None:
        continue  # skip [CLS] and [SEP]
    tag = ner_model.config.id2label[pid]
    print(f"{tok:<20} {tag}")


### What just happened?
- The reloaded model uses `config.id2label` (saved during training) to convert integer predictions back to tag strings — no need to pass `label_names` separately.
- Subword tokens from `##` continuation are still present in the output — in production you'd aggregate predictions to word level using `word_ids()` to avoid double-reporting.
- **This is the same logic as our `tokenize_and_align_labels` function, in reverse** — words split into subwords during encoding must be merged back during decoding.
- For production use, the `pipeline("ner", aggregation_strategy="word")` helper handles word-level aggregation automatically.


In [ ]:
# Challenge: Evaluate per-entity-type F1 breakdown
# Your solution here
#
# 1. Run predictions on the small_eval dataset using trainer.predict()
#    — it returns a PredictionOutput with .predictions and .label_ids
# 2. Convert predictions → tag strings (same logic as compute_metrics)
# 3. Call seqeval_metric.compute(predictions=pred_labels, references=true_labels)
# 4. Print per-entity-type precision, recall, and F1:
#    for entity_type in ['PER', 'ORG', 'LOC', 'MISC']:
#        print(entity_type, results[entity_type])
# 5. Which entity type has the lowest F1? Why might that be?
#    (Hint: consider how ambiguous MISC entities are compared to LOC/PER)

# predictions_output = trainer.predict(small_eval)
# ...


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| IOB2 tagging | B- starts a span, I- continues it, O is outside; adjacent same-type spans need B- at boundary |
| `is_split_into_words=True` | Tells the tokenizer input is word-tokenized, not raw strings |
| `word_ids()` | Maps each subword token position back to its source word index |
| Label `-100` | Ignored by `CrossEntropyLoss`; use it for continuation subwords and special tokens |
| `DataCollatorForTokenClassification` | Pads `labels` with `-100` alongside `input_ids` |
| seqeval | Entity-level (span-level) precision/recall/F1; stricter than token accuracy |
| `id2label` in config | Embed tag names in the saved model so inference doesn't need external label lists |

> **Tip:** Set the label for the first subword token of each word and use -100 for continuation tokens — PyTorch's CrossEntropyLoss ignores index -100 by default, so continuation tokens don't contribute to the loss.

---
## What's next
**Day 9** → Question Answering — Extractive QA with SQuAD-Style Models: instead of assigning a label to each token, predict two token positions (start and end) that form the answer span within the context.

Mark Day 8 complete in your [tracker](../index.html).
